In [1]:
import numpy as np
from itertools import combinations

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

from pyblock3.fcidump import FCIDUMP
from pyblock3.hamiltonian import Hamiltonian
from pyblock3.algebra.mpe import MPE
from pyblock3.algebra.symmetry import SZ
import pyblock3.algebra.ad as ad
ad.ENABLE_JAX = True                                   # before the ad submodules are imported
from pyblock3.algebra.ad.core import SparseTensor, SubTensor
from pyblock3.algebra.ad.mps import MPS

L, U, Na, CHI = 8, 2.0, 4, 20                          # 8 sites, half filling, small bond dim
np.set_printoptions(precision=4, suppress=True, linewidth=110)
print(f"Hubbard L={L}, U/t={U}, N=({Na},{Na});  DMRG bond dim {CHI}")

Hubbard L=8, U/t=2.0, N=(4,4);  DMRG bond dim 20


In [4]:
#Run small DMRG to compute the "trial" wfn
def hubbard_mpo(L, U, t=1.0, nelec=None, twos=0):
    h1e = np.zeros((L, L))
    for i in range(L-1):
        h1e[i, i+1] = h1e[i+1, i] = -t
    g2e = np.zeros((L,)*4)
    for i in range(L):
        g2e[i, i, i, i] = U
    fd = FCIDUMP(pg='c1', n_sites=L, n_elec=nelec if nelec is not None else 2*Na,
                 twos=twos, ipg=0, h1e=h1e, g2e=g2e)
    return Hamiltonian(fd, flat=True)

def run_dmrg(hamil, bdim, n_sweeps=12):
    mpo, _ = hamil.build_qc_mpo().compress(cutoff=1e-12)
    mps = hamil.build_mps(bdim)
    dmrg = MPE(mps, mpo, mps).dmrg(bdims=[bdim]*n_sweeps, noises=[1e-5]*5+[0],
                                   dav_thrds=[1e-10], iprint=0, n_sweeps=n_sweeps)
    return mps, float(dmrg.energies[-1])

hamil = hubbard_mpo(L, U)
mps_dmrg, E_dmrg = run_dmrg(hamil, CHI)
print(f"DMRG chi={CHI}:  E = {E_dmrg:.12f}   bond dims {mps_dmrg.show_bond_dims()}")

QC MPO site   0 / 8
QC MPO site   1 / 8
QC MPO site   2 / 8
QC MPO site   3 / 8
QC MPO site   4 / 8
QC MPO site   5 / 8
QC MPO site   6 / 8
QC MPO site   7 / 8
Time elapsed =      0.197 | E =   -6.162554251247084 | DE = 0.00E+00 | MDW = 8.32E-05 | MEM =  271 MB
Time sweep =      0.197 | Time davidson =      0.000 | Time decomp =      0.013
Time elapsed =      0.248 | E =   -6.224641750804915 | DE = 6.21E-02 | MDW = 1.10E-04 | MEM =  271 MB
Time sweep =      0.051 | Time davidson =      0.000 | Time decomp =      0.013
Time elapsed =      0.292 | E =   -6.225066681869114 | DE = 4.25E-04 | MDW = 1.11E-04 | MEM =  271 MB
Time sweep =      0.044 | Time davidson =      0.000 | Time decomp =      0.012
Time elapsed =      0.332 | E =   -6.225066681873288 | DE = 4.17E-12 | MDW = 1.11E-04 | MEM =  271 MB
Time sweep =      0.040 | Time davidson =      0.000 | Time decomp =      0.012
Time elapsed =      0.370 | E =   -6.225066083682082 | DE = 5.98E-07 | MDW = 1.11E-04 | MEM =  271 MB
Time sweep

In [ ]:
#Learning the pyblock3 index convention and checking it
def spin_occ(q):
    """(n_alpha, n_beta) of a site label: n = na + nb, 2Sz = na - nb."""
    return (int(q.n) + int(q.twos)) // 2, (int(q.n) - int(q.twos)) // 2

def local_index_map(hamil):
    """(n_alpha, n_beta) -> local index, straight from pyblock3's own site basis."""
    return {spin_occ(SZ.from_flat(int(code))): k for k, code in enumerate(hamil.basis[0])}

def flat_blocks(mps, i):
    """(q_labels, shape, data) for every block of site i, with PYTHON-int labels."""
    t = mps[i]
    for k in range(t.n_blocks):
        q = tuple(SZ.from_flat(int(x)) for x in t.q_labels[k])
        sh = tuple(int(x) for x in t.shapes[k])
        yield q, sh, np.asarray(t.data[t.idxs[k]:t.idxs[k+1]]).reshape(sh)

def sector_determinants(L, Na, Nb):
    """Every (occ_alpha, occ_beta) with the given filling, as 0/1 arrays."""
    occs = lambda N: [np.isin(np.arange(L), c).astype(int) for c in combinations(range(L), N)]
    return [(a, b) for a in occs(Na) for b in occs(Nb)]

def amplitudes(mps, dets, index_map):
    """<d|MPS> for each determinant, via pyblock3's own sliceable-tensor contraction."""
    sl = mps.to_non_flat().to_sliceable()
    return np.array([sl.amplitude([index_map[(int(a), int(b))] for a, b in zip(oa, ob)])
                     for oa, ob in dets])

LOCAL = local_index_map(hamil)
print("site basis in index order:")
for k, code in enumerate(hamil.basis[0]):
    q = SZ.from_flat(int(code))
    print(f"   index {k}: {q}   (n_alpha, n_beta) = {spin_occ(q)}")
print(f"-> local index = n_alpha + 2 n_beta  ({LOCAL})\n")

# verify by completeness on a spin-polarised random MPS: no DMRG, no ED
Lp, Nap, Nbp = 4, 3, 1
hp = hubbard_mpo(Lp, 4.0, nelec=Nap+Nbp, twos=Nap-Nbp)
mp = hp.build_mps(30)
dets_p, LOCAL_p = sector_determinants(Lp, Nap, Nbp), local_index_map(hp)

print(f"random MPS, L={Lp}, target N=({Nap},{Nbp}):  norm^2 = {mp.norm()**2:.10f}")
for name, m in [("map from hamil.basis", LOCAL_p)]:
    c = amplitudes(mp, dets_p, m)
    print(f"   {name}: sum_d |<d|MPS>|^2 = {(c**2).sum():.10f}")

site basis in index order:
   index 0: < N=0 SZ=0 PG=0 >   (n_alpha, n_beta) = (0, 0)
   index 1: < N=1 SZ=1/2 PG=0 >   (n_alpha, n_beta) = (1, 0)
   index 2: < N=1 SZ=-1/2 PG=0 >   (n_alpha, n_beta) = (0, 1)
   index 3: < N=2 SZ=0 PG=0 >   (n_alpha, n_beta) = (1, 1)
-> local index = n_alpha + 2 n_beta  ({(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3})

random MPS, L=4, target N=(3,1):  norm^2 = 0.0815619669
   map from hamil.basis: sum_d |<d|MPS>|^2 = 0.0815619669
   the other assignment: sum_d |<d|MPS>|^2 = 0.0000000000


/Users/fnappi/.trot/lib/python3.12/site-packages/pyblock3/algebra/symmetry.py:45: RuntimeWarning: overflow encountered in scalar subtract
  return SZ((x // 131072) % 16384 - 8192, (x // 8) % 16384 - 8192, x % 8)


In [6]:
#Expanding the MPS as an MSD
dets = sector_determinants(L, Na, Na)
c_dmrg = amplitudes(mps_dmrg, dets, LOCAL)
w = c_dmrg**2
order = np.argsort(-w)
print(f"{len(dets)} determinants in the ({Na},{Na}) sector,"
      f" {(np.abs(c_dmrg) > 1e-10).sum()} with |c| > 1e-10")
print(f"  completeness: sum_d |c_d|^2 = {w.sum():.12f}   (MPS norm^2 = {mps_dmrg.norm()**2:.12f})")
print(f"  {int(np.searchsorted(np.cumsum(w[order]), 0.99)) + 1} determinants carry 99% of the weight\n")
sym = {(0, 0): '.', (1, 0): 'u', (0, 1): 'd', (1, 1): '2'}
print("  leading determinants (. empty, u up, d down, 2 doubly occupied)")
for j in order[:6]:
    oa, ob = dets[j]
    print(f"    {''.join(sym[(int(a), int(b))] for a, b in zip(oa, ob))}"
          f"   c = {c_dmrg[j]:+.8f}   weight {w[j]:.6f}")

4900 determinants in the (4,4) sector, 4500 with |c| > 1e-10
  completeness: sum_d |c_d|^2 = 1.000000000000   (MPS norm^2 = 1.000000000000)
  2395 determinants carry 99% of the weight

  leading determinants (. empty, u up, d down, 2 doubly occupied)
    dudududu   c = -0.14589159   weight 0.021284
    udududud   c = -0.14589159   weight 0.021284
    uddududu   c = +0.10444464   weight 0.010909
    duududud   c = +0.10444464   weight 0.010909
    udududdu   c = +0.10443845   weight 0.010907
    dududuud   c = +0.10443845   weight 0.010907


In [ ]:
#Using HF as a reference walker and computing the first overlap
def rhf_orbitals(L, Na, t=1.0):
    h = np.zeros((L, L))
    for i in range(L-1):
        h[i, i+1] = h[i+1, i] = -t
    return np.linalg.eigh(h)[1][:, :Na], h

def interleaving_sign(occ_a, occ_b):
    """(-1)^K relating 'all alpha then all beta' to the interleaved lattice order."""
    ra, rb = np.where(occ_a == 1)[0], np.where(occ_b == 1)[0]
    return (-1)**sum(int((rb < i).sum()) for i in ra)

def hf_amplitude(Ca, Cb, occ_a, occ_b):
    """<d|HF> from determinant minors."""
    ra, rb = np.where(occ_a == 1)[0], np.where(occ_b == 1)[0]
    if len(ra) != Ca.shape[1] or len(rb) != Cb.shape[1]:
        return 0.0
    return (interleaving_sign(occ_a, occ_b)
            * np.linalg.det(Ca[ra, :]) * np.linalg.det(Cb[rb, :]))

Ca, h1 = rhf_orbitals(L, Na)
Cb = Ca
E_hf = 2*np.sum(np.linalg.eigvalsh(h1)[:Na]) + U*np.sum((Ca**2).sum(1) * (Cb**2).sum(1))
c_hf = np.array([hf_amplitude(Ca, Cb, oa, ob) for oa, ob in dets])
print(f"E_HF = {E_hf:.10f}   E_DMRG = {E_dmrg:.10f}")
print(f"completeness of the minor amplitudes: sum_d |<d|HF>|^2 = {(c_hf**2).sum():.12f}")
route1 = float(c_hf @ c_dmrg)
print(f"route 1:  <HF|MSD> = sum_d <HF|d> c_d = {route1:+.12f}   ({len(dets)} determinants)")

E_HF = -5.5175409663   E_DMRG = -6.2250660420
completeness of the minor amplitudes: sum_d |<d|HF>|^2 = 1.000000000000
route 1:  <HF|MSD> = sum_d <HF|d> c_d = -0.901259586542   (4900 determinants)


In [ ]:
#Now we turn the HF walker into an MPS, one spin at the time, and the combining the 2 spinless MPS into one spinful one
PHYS = np.array([[li % 2, li // 2] for li in range(4)])     # local index -> (n_alpha, n_beta)

def V_hat(theta):
    """The spinless two-site gate, Eq. (9): one spin channel needs nothing more."""
    c, s = jnp.cos(theta), jnp.sin(theta)
    g = jnp.eye(4).at[1, 1].set(c).at[1, 2].set(s).at[2, 1].set(-s).at[2, 2].set(c)
    return g.reshape(2, 2, 2, 2)

def gmps_channel(C, eps=1e-10):
    """Fishman-White on one spin channel: occupations, gates, and the ROTATED orbitals."""
    U_, n = jnp.asarray(C, float), C.shape[0]
    occ, gates, Bs = np.zeros(n, int), [], []
    for k in range(n-1):
        Lam = U_ @ U_.T
        for B in range(2, n - k + 1):
            nb, W = jnp.linalg.eigh(Lam[k:k+B, k:k+B])
            if float(jnp.minimum(nb[0], 1.0 - nb[-1])) < eps:
                break
        if nb[0] <= 1.0 - nb[-1]: v, occ[k] = W[:, 0], 0
        else:                     v, occ[k] = W[:, -1], 1
        Bs.append(B)
        for j in range(B-1, 0, -1):
            theta = jnp.arctan2(v[j], v[j-1])
            c, s = jnp.cos(theta), jnp.sin(theta)
            v = v.at[j-1].set(c*v[j-1] + s*v[j]).at[j].set(0.0)
            U_ = U_.at[k+j-1].set(c*U_[k+j-1] + s*U_[k+j]).at[k+j].set(
                -s*U_[k+j-1] + c*U_[k+j])
            gates.append((k+j-1, theta))
    occ[n-1] = int(round(float(U_[n-1] @ U_[n-1])))
    assert occ.sum() == C.shape[1], f"particle number lost in the {C.shape[1]}-electron channel"
    return occ, gates, Bs, np.asarray(U_)

def apply_gate_1d(ts, qn, p, theta, chi_max=None, cutoff=1e-12):
    """Spinless gate on (p, p+1), then an SVD split inside each particle-number sector."""
    T = jnp.einsum("xypq,apqb->axyb", V_hat(theta), jnp.tensordot(ts[p], ts[p+1], axes=1))
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl*2, 2*Dr)
    rc = (qn[p][:, None] + np.arange(2)[None, :]).ravel()
    cc = (qn[p+2][None, :] - np.arange(2)[:, None]).ravel()
    sectors = []
    for nm in sorted(set(rc.tolist()) & set(cc.tolist())):
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        u, sv, vt = jnp.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
        sectors.append((nm, r, c, u, sv, vt))
    alls = np.concatenate([np.asarray(x[4]) for x in sectors])
    thr = cutoff * max(alls.max(), 1e-300)
    if chi_max is not None and int((alls > thr).sum()) > chi_max:
        thr = np.sort(alls)[::-1][:chi_max][-1]
    A, Bc, qm = [], [], []
    for nm, r, c, u, sv, vt in sectors:
        k = int((np.asarray(sv) >= thr).sum())
        if k == 0:
            continue
        A.append(jnp.zeros((Dl*2, k)).at[r].set(u[:, :k]))
        Bc.append(jnp.zeros((k, 2*Dr)).at[:, c].set(sv[:k, None] * vt[:k]))
        qm += [nm] * k
    ts, qn = list(ts), list(qn)
    ts[p] = jnp.concatenate(A, axis=1).reshape(Dl, 2, -1)
    ts[p+1] = jnp.concatenate(Bc, axis=0).reshape(-1, 2, Dr)
    qn[p+1] = np.array(qm, int)
    return ts, qn

def channel_mps(C, chi_max=None, cutoff=1e-13):
    """One spin species -> a d=2 MPS, bond charges, occupations, max block size, gauge sign."""
    occ, gates, Bs, U_rot = gmps_channel(C)
    ts, qn = [], [np.zeros(1, int)]
    for o in occ:
        ts.append(jax.nn.one_hot(int(o), 2).reshape(1, 2, 1))
        qn.append(qn[-1] + int(o))
    for p, theta in reversed(gates):
        ts, qn = apply_gate_1d(ts, qn, p, theta, chi_max, cutoff)
    sgn = int(round(float(np.linalg.det(U_rot[np.where(occ == 1)[0], :]))))
    return ts, qn, occ, max(Bs), sgn

def combine_channels(Aa, qna, Ab, qnb):
    """Two d=2 MPSs -> one d=4 MPS in the interleaved order (0a, 0b, 1a, 1b, ...).

    The (-1)^(n_alpha * l_beta) factor is the interleaving sign, written site by site with
    the beta bond label serving as the running count of beta electrons to the left.
    """
    tensors, qn = [], [np.zeros((1, 2), int)]
    for i in range(len(Aa)):
        Dal, _, Dar = Aa[i].shape
        Dbl, _, Dbr = Ab[i].shape
        out = jnp.zeros((Dal, Dbl, 4, Dar, Dbr))
        for na in (0, 1):
            sgn = (-1.0) ** (na * qnb[i])                      # diagonal on the beta bond
            for nb in (0, 1):
                out = out.at[:, :, na + 2*nb, :, :].set(
                    jnp.einsum("ar,b,bs->abrs", Aa[i][:, na, :], sgn, Ab[i][:, nb, :]))
        tensors.append(out.reshape(Dal*Dbl, 4, Dar*Dbr))
        qn.append(np.stack([np.repeat(qna[i+1], Dbr), np.tile(qnb[i+1], Dar)], axis=1))
    return tensors, qn

def channel_amplitude(A, occ_bits):
    """<d|MPS> for a single d=2 channel: a bond-dimension-1 contraction."""
    v = np.ones((1, 1))
    for Ai, o in zip(A, occ_bits):
        v = v @ np.asarray(Ai[:, int(o), :])
    return float(v[0, 0])

# --- one channel at a time, each checked against its own spinless minor formula
Aa, qna, occ_a, Ba, sa = channel_mps(Ca)
Ab, qnb, occ_b, Bb, sb = channel_mps(Cb)
probe1 = [np.isin(np.arange(L), c).astype(int) for c in list(combinations(range(L), Na))[:40]]
for name, A, C_, B_, sg in (("alpha", Aa, Ca, Ba, sa), ("beta ", Ab, Cb, Bb, sb)):
    dims = [t.shape[0] for t in A] + [A[-1].shape[2]]
    err = max(abs(sg*channel_amplitude(A, o) - np.linalg.det(C_[np.where(o == 1)[0], :]))
              for o in probe1)
    print(f"{name} channel: d=2, bond dims {dims}, max B {B_}, gauge {sg:+d};"
          f"  max |<d|MPS> - det C[d]| = {err:.1e}")

tensors, qn = combine_channels(Aa, qna, Ab, qnb)
sgn = sa * sb                       # the interleaving sign now lives inside combine_channels
print(f"\ncombined: d=4, bond dims {[t.shape[0] for t in tensors] + [tensors[-1].shape[2]]}"
      f"  = products of the two channels;  gauge {sgn:+d}")

alpha channel: d=2, bond dims [1, 2, 4, 8, 16, 8, 4, 2, 1], max B 5, gauge +1;  max |<d|MPS> - det C[d]| = 1.4e-15
beta  channel: d=2, bond dims [1, 2, 4, 8, 16, 8, 4, 2, 1], max B 5, gauge +1;  max |<d|MPS> - det C[d]| = 1.4e-15



combined: d=4, bond dims [1, 4, 16, 64, 256, 64, 16, 4, 1]  = products of the two channels;  gauge +1


In [ ]:
#Now let's compare the overlaps
Q = lambda na, nb: SZ(int(na) + int(nb), int(na) - int(nb), 0)

def to_pyblock3(tensors, qn):
    out = []
    for i, A in enumerate(tensors):
        blocks = []
        for nl in sorted(set(map(tuple, qn[i].tolist()))):
            rows = np.where((qn[i] == np.array(nl)).all(1))[0]
            for li in range(4):
                nr = tuple(np.array(nl) + PHYS[li])
                cols = np.where((qn[i+1] == np.array(nr)).all(1))[0]
                if len(cols) == 0:
                    continue
                data = A[np.ix_(rows, [li], cols)]
                if float(jnp.abs(data).max()) == 0.0:
                    continue
                blocks.append(SubTensor(data=data, q_labels=(Q(*nl), Q(*PHYS[li]), Q(*nr))))
        out.append(SparseTensor(blocks=blocks))
    return MPS(tensors=out)

def flat_to_jax(mps):
    """pyblock3 flat MPS -> ad/jax MPS, labels decoded as Python ints."""
    return MPS(tensors=[SparseTensor(blocks=[SubTensor(data=jnp.asarray(d), q_labels=q)
                                             for q, _, d in flat_blocks(mps, i)])
                        for i in range(mps.n_sites)])

def det_mps(occ_a, occ_b):
    """|d> as a bond-dimension-1 ad/jax MPS, labelled like to_pyblock3."""
    ts, nl = [], np.zeros(2, int)
    for na, nb in zip(occ_a, occ_b):
        nr = nl + np.array([int(na), int(nb)])
        ts.append(SparseTensor(blocks=[SubTensor(data=jnp.ones((1, 1, 1)),
                                                 q_labels=(Q(*nl), Q(int(na), int(nb)), Q(*nr)))]))
        nl = nr
    return MPS(tensors=ts)

mps_hf = to_pyblock3(tensors, qn)
mps_dmrg_jax = flat_to_jax(mps_dmrg)
print(f"MPS_HF   : {mps_hf.show_bond_dims()}   norm {mps_hf.norm():.10f}")
print(f"MPS_DMRG : {mps_dmrg_jax.show_bond_dims()}   norm {mps_dmrg_jax.norm():.10f}")
print(f"blocks are jax arrays: "
      f"{all(isinstance(l, jax.Array) for l in jax.tree_util.tree_leaves(mps_hf)[:-1])}\n")

probe = [dets[j] for j in order[:4]] + [(occ_a, occ_b)]
print("  <d|MPS_HF> against the minor formula")
worst = 0.0
for oa, ob in probe:
    got = sgn * float(mps_hf.dot(det_mps(oa, ob)))
    ref = hf_amplitude(Ca, Cb, oa, ob)
    worst = max(worst, abs(got - ref))
    print(f"    {''.join(sym[(int(a), int(b))] for a, b in zip(oa, ob))}"
          f"   MPS {got:+.12f}   minors {ref:+.12f}")
print(f"  worst difference {worst:.1e}\n")

route2 = sgn * float(mps_hf.dot(mps_dmrg_jax))
print(f"  route 1   sum_d <HF|d> c_d      = {route1:+.12f}")
print(f"  route 2   <MPS_HF|MPS_DMRG>     = {route2:+.12f}")
print(f"  difference                      = {abs(route1 - route2):.2e}")

MPS_HF   : 1|4|16|64|256|64|16|4|1   norm 1.0000000000


MPS_DMRG : 1|4|16|20|20|20|16|4|1   norm 1.0000000000
blocks are jax arrays: True

  <d|MPS_HF> against the minor formula


    udududud   MPS +0.062500000000   minors +0.062500000000
    dudududu   MPS +0.062500000000   minors +0.062500000000
    duududud   MPS -0.046449474853   minors -0.046449474853
    uddududu   MPS -0.046449474853   minors -0.046449474853
    2..2.2.2   MPS +0.046449474853   minors +0.046449474853
  worst difference 2.1e-16



  route 1   sum_d <HF|d> c_d      = +0.901259586410
  route 2   <MPS_HF|MPS_DMRG>     = +0.901259586410
  difference                      = 2.44e-15


In [ ]:
#How much can we truncate the MPS walker?
full = sgn * float(mps_hf.dot(mps_dmrg_jax))
print(f"as built:  chi = 256   <MPS_HF|MPS_DMRG> = {full:+.12f}\n")
print("  compression            chi    fidelity vs uncompressed    overlap        error")
for kw in [dict(cutoff=1e-12), dict(max_bond_dim=64), dict(max_bond_dim=32), dict(max_bond_dim=8)]:
    r = mps_hf.copy().compress(**kw)
    m = r[0] if isinstance(r, tuple) else r
    chi = max(int(x) for x in m.show_bond_dims().split('|'))
    fid = abs(float(m.dot(mps_hf))) / (m.norm() * mps_hf.norm())
    ov = sgn * float(m.dot(mps_dmrg_jax)) / m.norm()
    print(f"  {str(kw):22s} {chi:4d}      {fid:.12f}        {ov:+.9f}   {abs(ov-full):.1e}")
print("\ncutoff=1e-12 removes nothing: 256 is the true rank, not slack.")

as built:  chi = 256   <MPS_HF|MPS_DMRG> = +0.901259586410

  compression            chi    fidelity vs uncompressed    overlap        error


  {'cutoff': 1e-12}       256      1.000000000000        +0.901259586   2.9e-15


  {'max_bond_dim': 64}     64      0.999477324049        +0.900709334   5.5e-04


  {'max_bond_dim': 32}     32      0.999337317544        +0.900533903   7.3e-04


  {'max_bond_dim': 8}       8      0.958622529303        +0.844779286   5.6e-02

cutoff=1e-12 removes nothing: 256 is the true rank, not slack.


In [ ]:
#The mixed energy estimator: <psi_T|H|phi> with the DMRG trial state and the MPS walker
mpo_h, _ = hamil.build_qc_mpo().compress(cutoff=1e-12)

def h_on_trial(mpo, mps_T, cutoff=1e-12, max_bond_dim=-1):
    """H|psi_T> as an ad/jax MPS, plus the flat one for the determinant check.

    psi_T never moves, so the MPO is applied once, up front. What is left for the walker is
    a single MPS-MPS contraction -- the same object, and the same cost, as the overlap
    above, and just as differentiable.
    """
    hket, _ = (mpo @ mps_T).compress(cutoff=cutoff, max_bond_dim=max_bond_dim)
    assert float(hket.const) == 0.0, "constant term not folded into the tensors"
    return flat_to_jax(hket), hket

def mixed_energy(mps_walker, h_psi_T, mps_T, gauge=1.0):
    """E_mix = <psi_T|H|phi> / <psi_T|phi>;  returns (E_mix, numerator, denominator).

    H is Hermitian and everything here is real, so <psi_T|H|phi> = <H psi_T|phi>: both
    numerator and denominator are plain overlaps against the walker, differing only in
    which fixed bra they contract with. `gauge` is the walker's overall sign (sa*sb from
    channel_mps), which cancels in the ratio but not in the bare numerator.
    """
    num = gauge * mps_walker.dot(h_psi_T)
    den = gauge * mps_walker.dot(mps_T)
    return num / den, num, den

h_psi_T, h_psi_flat = h_on_trial(mpo_h, mps_dmrg)
print(f"H|psi_T> : {h_psi_flat.show_bond_dims()}   (psi_T was {mps_dmrg.show_bond_dims()},"
      f" MPO {mpo_h.show_bond_dims()})")

# 1. the construction, against pyblock3's own expectation value
E_var = float(mps_dmrg_jax.dot(h_psi_T) / mps_dmrg_jax.dot(mps_dmrg_jax))
print(f"  <psi_T|H|psi_T>/<psi_T|psi_T> = {E_var:.12f}   (jax route)")
print(f"                                = {MPE(mps_dmrg, mpo_h, mps_dmrg).expectation:.12f}"
      f"   (pyblock3 MPE)")
print(f"  the sweep energy above, {E_dmrg:.12f}, is the two-site value before the last"
      f" decimation\n")

# 2. the numerator, the same two routes as the overlap: determinants vs MPS contraction
c_hpsi = amplitudes(h_psi_flat, dets, LOCAL)            # <d|H|psi_T> for every determinant
E_mix, num, den = mixed_energy(mps_hf, h_psi_T, mps_dmrg_jax, sgn)
print(f"  route 1   sum_d <HF|d> <d|H|psi_T>  = {float(c_hf @ c_hpsi):+.12f}")
print(f"  route 2   <MPS_HF|H psi_T>          = {float(num):+.12f}")
print(f"  difference                          = {abs(float(c_hf @ c_hpsi) - float(num)):.2e}\n")

# 3. the estimator
print(f"  <psi_T|H|phi> = {float(num):+.12f}")
print(f"  <psi_T|phi>   = {float(den):+.12f}")
print(f"  E_mix         = {E_mix:.12f}     vs  E_HF {E_hf:.12f},  E_var[psi_T] {E_var:.12f}")

# 4. the estimator is exact for any walker once psi_T is: repeat with a full-rank trial state
mps_ex = hamil.build_mps(256)                   # the sweep of run_dmrg, reusing mpo_h, silent
MPE(mps_ex, mpo_h, mps_ex).dmrg(bdims=[256]*8, noises=[1e-5]*4+[0], dav_thrds=[1e-10],
                                iprint=-1, n_sweeps=8)
h_psi_ex, _ = h_on_trial(mpo_h, mps_ex)
mps_ex_jax = flat_to_jax(mps_ex)
E_ex = float(mps_ex_jax.dot(h_psi_ex) / mps_ex_jax.dot(mps_ex_jax))
print(f"\nwith a full-rank trial state ({mps_ex.show_bond_dims()}, E = {E_ex:.12f});"
      f" its global sign is its own, and cancels in E_mix")
oa, ob = dets[order[0]]
for name, walker, g in (("MPS_HF", mps_hf, sgn),
                        (f"|{''.join(sym[(int(a), int(b))] for a, b in zip(oa, ob))}>",
                         det_mps(oa, ob), 1.0)):
    E_m, n_, d_ = mixed_energy(walker, h_psi_ex, mps_ex_jax, g)
    print(f"  phi = {name:10s} <psi_T|H|phi> {float(n_):+.9f}   <psi_T|phi> {float(d_):+.9f}"
          f"   E_mix {E_m:.12f}   E_mix - E_0 {E_m - E_ex:+.1e}")
print("  the walker drops out entirely: the mixed estimator is exact when psi_T is.")

## 9. Making JAX earn its keep: freeze the plan, replay it

The channel builds above run on `jnp` but are not compiled, and that is deliberate: the shapes
are data. Every sector-blocked split has its own size and the bonds grow as gates are applied,
so `jit` would recompile constantly. Measured on one channel build at $L=16$: 251 SVD calls,
**33 distinct shapes**, and on a cold run **96% of the wall time is XLA compilation** rather
than arithmetic. Worse, warm, `jnp` is 28–45$\times$ *slower* than the same code in NumPy here,
because these are $16\times16$ SVDs and per-op dispatch dominates.

The fix that works is the **plan/replay** split. Run the build once on the host and record what
it decided — the sector row/column lists and how many singular values survived, hence every
array shape. The replay then takes only the angles and has nothing left to decide, so the whole
channel build traces to *one* XLA program. This is exact, not an approximation: the discrete
choices are locally constant in the angles, so nudging $\theta$ reproduces the same sectors and
ranks, and the frozen plan is valid in a neighbourhood. Re-plan when the state moves far enough
to change a rank.

(The other option is to pad every bond to a fixed $\chi$ and compile one kernel. It is simpler
and its compile cost does not grow with $L$, but it wastes arithmetic on full-width SVDs — and
it has a trap: the truncation must **mask**, not keep the largest $\chi$ by count. These local
splits are never canonicalised, so the matrix's singular values are not the state's Schmidt
values and its rank can exceed the physical one; slicing by count silently loses real weight.)

In [10]:
import time
from functools import partial

def record_channel_plan(C, cutoff=1e-13):
    """Run one channel build on the host, recording every shape it chooses.

    Returns (occ, thetas, plan, qn, gauge): everything the replay needs, with the plan
    holding per-gate (p, Dl, Dr, sectors) and each sector (rows, cols, retained rank).
    """
    occ, gates, Bs, U_rot = gmps_channel(C)
    n = len(occ)
    ts = [np.eye(2)[int(o)].reshape(1, 2, 1) for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    plan = []
    for p, theta in reversed(gates):
        g = np.asarray(V_hat(theta))
        T = np.einsum("xypq,apqb->axyb", g, np.tensordot(ts[p], ts[p+1], axes=1))
        Dl, _, _, Dr = T.shape
        M = T.reshape(Dl*2, 2*Dr)
        rc = (qn[p][:, None] + np.arange(2)[None, :]).ravel()
        cc = (qn[p+2][None, :] - np.arange(2)[:, None]).ravel()
        secs, A, Bc, qm = [], [], [], []
        for nm in sorted(set(rc.tolist()) & set(cc.tolist())):
            r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
            u, s, vt = np.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
            k = int((s > cutoff * max(s.max(), 1e-300)).sum())
            if k == 0:
                continue
            secs.append((tuple(r), tuple(c), k))
            a = np.zeros((Dl*2, k)); a[r] = u[:, :k]
            b = np.zeros((k, 2*Dr)); b[:, c] = s[:k, None] * vt[:k]
            A.append(a); Bc.append(b); qm += [nm] * k
        plan.append((p, Dl, Dr, tuple(secs)))
        ts[p] = np.hstack(A).reshape(Dl, 2, -1)
        ts[p+1] = np.vstack(Bc).reshape(-1, 2, Dr)
        qn[p+1] = np.array(qm, int)
    sgn = int(round(float(np.linalg.det(U_rot[np.where(occ == 1)[0], :]))))
    thetas = jnp.array([float(t) for _, t in reversed(gates)])
    return occ, thetas, plan, qn, sgn

def make_channel_replay(occ, plan):
    """Compile one channel's build for a frozen plan: angles in, tensors out."""
    def run(thetas):
        ts = [jax.nn.one_hot(int(o), 2).reshape(1, 2, 1) for o in occ]
        for (p, Dl, Dr, secs), theta in zip(plan, thetas):
            T = jnp.einsum("xypq,apqb->axyb", V_hat(theta),
                           jnp.tensordot(ts[p], ts[p+1], axes=1))
            M = T.reshape(Dl*2, 2*Dr)
            A, Bc = [], []
            for r, c, k in secs:
                r, c = np.array(r), np.array(c)
                u, s, vt = jnp.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
                A.append(jnp.zeros((Dl*2, k)).at[r].set(u[:, :k]))
                Bc.append(jnp.zeros((k, 2*Dr)).at[:, c].set(s[:k, None] * vt[:k]))
            ts = list(ts)
            ts[p] = jnp.concatenate(A, axis=1).reshape(Dl, 2, -1)
            ts[p+1] = jnp.concatenate(Bc, axis=0).reshape(-1, 2, Dr)
        return ts
    return jax.jit(run)

# plan both channels, replay them, glue, and check nothing moved
occ_a2, th_a, plan_a, qn_a2, sa2 = record_channel_plan(Ca)
occ_b2, th_b, plan_b, qn_b2, sb2 = record_channel_plan(Cb)
rep_a, rep_b = make_channel_replay(occ_a2, plan_a), make_channel_replay(occ_b2, plan_b)

t0 = time.perf_counter()
Aa_r = jax.block_until_ready(rep_a(th_a)); Ab_r = jax.block_until_ready(rep_b(th_b))
t_cold = time.perf_counter() - t0
t0 = time.perf_counter()
Aa_r = jax.block_until_ready(rep_a(th_a)); Ab_r = jax.block_until_ready(rep_b(th_b))
t_warm = time.perf_counter() - t0
t0 = time.perf_counter(); channel_mps(Ca); channel_mps(Cb); t_dyn = time.perf_counter() - t0

tensors_r, qn_r = combine_channels(Aa_r, qn_a2, Ab_r, qn_b2)
mps_hf_r = to_pyblock3(tensors_r, qn_r)
route2_r = sa2 * sb2 * float(mps_hf_r.dot(mps_dmrg_jax))
print(f"plan: {len(plan_a)} + {len(plan_b)} gates,"
      f" {sum(len(x[3]) for x in plan_a) + sum(len(x[3]) for x in plan_b)} sector splits")
print(f"  gauge signs unchanged: {(sa2, sb2) == (sa, sb)}"
      f"   bond dims {mps_hf_r.show_bond_dims()}")
print(f"  <MPS_HF|MPS_DMRG> from the replayed build = {route2_r:+.12f}")
print(f"                            route 1 (above) = {route1:+.12f}"
      f"   difference {abs(route2_r - route1):.1e}")
print(f"\nboth channels, dynamic jnp {t_dyn*1e3:8.1f} ms"
      f"   |   replay {t_warm*1e3:6.2f} ms after {t_cold*1e3:6.0f} ms of compilation")
print(f"  speedup once compiled: {t_dyn/t_warm:.0f}x"
      f"   (break-even after about {int(t_cold/max(t_dyn-t_warm, 1e-9))} builds)")

plan: 16 + 16 gates, 92 sector splits
  gauge signs unchanged: True   bond dims 1|4|16|64|256|64|16|4|1
  <MPS_HF|MPS_DMRG> from the replayed build = +0.901259586410
                            route 1 (above) = +0.901259586410   difference 2.4e-15

both channels, dynamic jnp    154.7 ms   |   replay   0.78 ms after    864 ms of compilation
  speedup once compiled: 198x   (break-even after about 5 builds)


The replayed build reproduces the overlap to the same 12 digits, and once compiled it is two
orders of magnitude faster than the dynamic route. The compile cost is real and grows with the
gate count, since the trace is unrolled — so the rule is:

* **one-shot, plan changes every time** → NumPy (or padding, if it has to be on device);
* **same plan reused** — gradients, a $\delta$ sweep, an orbital optimisation → replay;
* **dynamic `jnp`** → only because it reads like the maths, which is why the sections above
  use it.

The sweep that *produces* the angles has the same structure — its block sizes and
occupied/empty branches are discrete decisions belonging in the plan.
`fishman_white_tutorial_jax.ipynb` does that half, including the two `custom_jvp` rules needed
to differentiate through the near-degenerate `eigh`.